# LLM にさまざまなタスクを解かせる

プロンプトを変えて、同じ LLM に機械翻訳・感情分析・情報抽出・品詞タグ付けをさせます。
各セルのプロンプトをそのまま送り、返ってきた回答を確認します。
まず Gemini API で試し、その後、発展としてローカル LLM でも同じプロンプトを試します。

## API で LLM を使う

API を使うと、インターネット経由でモデルに入力を送り、回答を受け取ることができます。
[Google AI Studio](https://aistudio.google.com/apikey) で Gemini API キーを取得してください。
次のセルで API 用のライブラリをインストールします。

In [1]:
!pip install -q google-genai  # Gemini API の呼び出し

キーは次の入力欄に貼り付けます。入力した文字は表示されず、コードにも保存されません。

ここでは Gemini を使用します。
API キーをセットする必要がありますが、少量のリクエストなら無料で使用できるようです。
API の利用枠・料金は [Google AI Studio](https://aistudio.google.com/) で確認してください。

In [3]:
import os
from getpass import getpass

from google import genai

api_key = getpass("Gemini API キーを入力: ")
client = genai.Client(api_key=api_key)
API_MODEL_NAME = "gemini-flash-latest"

Gemini API キーを入力: ··········


これで API を呼び出す準備ができました。
以下では `prompt` に指示と入力文を書き、`print(response.text)` で回答を表示します。
同じプロンプトでも、実行するたびに回答が変わることがあります。

### 機械翻訳

日本語を英語に翻訳させます。

In [4]:
prompt = """日本語を英語に翻訳してください。翻訳だけを返してください。

昨日は雨でしたが、今日は晴れています。"""
response = client.models.generate_content(model=API_MODEL_NAME, contents=prompt)
print(response.text)

It rained yesterday, but it is sunny today.


### 感情分析

文の感情を 3 種類のラベルから選ばせます。

In [5]:
prompt = """文の感情を「肯定」「否定」「中立」のいずれか1語で答えてください。

映画は長かったけれど、最後まで夢中になりました。"""
response = client.models.generate_content(model=API_MODEL_NAME, contents=prompt)
print(response.text)

肯定


### 情報抽出

文から人名と場所を取り出し、JSON で答えさせます。JSON は、項目名と値を組にして書くデータ形式です。

In [6]:
prompt = """文中の人名と場所を抽出し、{"person": "人名", "place": "場所"} のJSONだけを返してください。

鈴木花子は京都を訪れました。"""
response = client.models.generate_content(model=API_MODEL_NAME, contents=prompt)
print(response.text)

{"person": "鈴木花子", "place": "京都"}


### 品詞タグ付け

文を単語に分け、それぞれの品詞を答えさせます。品詞の分け方には違いがあるため、ここでは使う品詞と分割の規則も指示します。

In [7]:
prompt = """文を単語に分けて「単語/品詞」を空白で区切って返してください。
品詞は名詞・助詞・動詞・記号を使い、動詞の活用語尾は分けません。

犬が走る。"""
response = client.models.generate_content(model=API_MODEL_NAME, contents=prompt)
print(response.text)

犬/名詞 が/助詞 走る/動詞 。/記号


## 発展：ローカル LLM で同じプロンプトを試す

ここからは、手元で動かす公開モデルを使います。API キーは使いません。

まずローカル推論用のライブラリをインストールします。

In [8]:
!pip install -q transformers  # モデルの読み込みと文章生成
!pip install -q torch torchvision # モデルを動かすためのライブラリ

`transformers` というライブラリを用いて、パラメータが公開されているモデルをダウンロードして使用します。
約 5 億パラメータです。API 経由で使用できるモデルに比べたら、かなりパラメータ数が小さいため、比較的性能は低いです。

In [9]:
import torch
from transformers import pipeline

torch.manual_seed(42)

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct", # Alibaba の開発する LLM
    device=0 if torch.cuda.is_available() else -1, # GPU が存在すればそれを使用し、なければ CPU を用いる
    max_new_tokens=128, # 最大出力トークン
    do_sample=False, # 毎回もっとも確率の高い次のトークンを選ぶ設定
    return_full_text=False, # 入力を除いた回答だけを返す設定
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


モデルを読み込めたら、同じプロンプトを一つずつ送ります。
`role="user"` はユーザーの入力であることを表し、`outputs[0]["generated_text"]` に回答が入ります。

### 機械翻訳

日本語を英語に翻訳させます。

In [10]:
prompt = """日本語を英語に翻訳してください。翻訳だけを返してください。

昨日は雨でしたが、今日は晴れています。"""
outputs = generator([{"role": "user", "content": prompt}])
print(outputs[0]["generated_text"])

Yesterday was rainy, but today is sunny.


### 感情分析

文の感情を 3 種類のラベルから選ばせます。

In [11]:
prompt = """文の感情を「肯定」「否定」「中立」のいずれか1語で答えてください。

映画は長かったけれど、最後まで夢中になりました。"""
outputs = generator([{"role": "user", "content": prompt}])
print(outputs[0]["generated_text"])

肯定


### 情報抽出

文から人名と場所を取り出し、JSON で答えさせます。JSON は、項目名と値を組にして書くデータ形式です。

In [13]:
prompt = """文中の人名と場所を抽出し、{"person": "人名", "place": "場所"} のJSONだけを返してください。

鈴木花子は京都を訪れました。"""
outputs = generator([{"role": "user", "content": prompt}])
print(outputs[0]["generated_text"])

```json
{
  "person": "鈴木花子",
  "place": "京都"
}
```


### 品詞タグ付け

文を単語に分け、それぞれの品詞を答えさせます。

Qwen2.5-0.5B-Instruct は日本語能力が高くなく、うまく答えてくれません。

In [15]:
prompt = """文を単語に分けて「単語/品詞」を空白で区切って返してください。
品詞は名詞・助詞・動詞・記号を使い、動詞の活用語尾は分けません。

犬が走る。"""
outputs = generator([{"role": "user", "content": prompt}])
print(outputs[0]["generated_text"])

品詞: 动詞

1. 狗が走る。 (動詞)
